# NE-Time foundation training — Colab fallback

Runs `train_foundation.py` on a Colab GPU when HPC is unavailable. Mirrors `scripts/train_foundation.sh` exactly — same flags, same defaults.

**Before running this notebook**: the code cell below `git clone`s from GitHub, so any local changes (e.g. the `--proj_hidden` flag) must be **committed and pushed** to the branch first:
```bash
git add -A && git commit -m "..."
git push -u origin <branch-name>
```
If the repo is private, replace `REPO_URL` below with `https://<GITHUB_TOKEN>@github.com/Tulsani/NE-Time.git` (a [fine-grained personal access token](https://github.com/settings/tokens) with read access is enough).

**Runtime**: Runtime menu -> Change runtime type -> GPU (T4 is fine, this model is only ~254K params). Colab free tier disconnects after ~90 min idle / 12h max — this notebook mounts Google Drive and points `--output_dir` directly at it, so checkpoints/logs/results persist through a disconnect and you can resume by just re-running from the top.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

## 1. Clone the branch

In [ ]:
REPO_URL = "https://github.com/Tulsani/NE-Time.git"  # prepend a token if private, see note above
BRANCH   = "ftr/mini-modal-foundational"               # change to whichever branch has the code you want

import os
if os.path.isdir("NE-Time"):
    %cd NE-Time
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} NE-Time
    %cd NE-Time

!git log --oneline -5

## 2. Install dependencies

torch/numpy/scikit-learn already ship with Colab; only `huggingface_hub` (for Weather/Exchange/ECL/Traffic) needs installing. Unlike HPC compute nodes, Colab has open internet, so this works directly — no separate login-node download step needed.

In [ ]:
!pip install -q huggingface_hub

## 3. Mount Google Drive (persistent outputs)

Every run's checkpoint/log/results files land directly in Drive via `--output_dir`, so a Colab disconnect mid-run doesn't lose anything — `train_foundation.py` already writes the best checkpoint to disk after every improving epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ne-time-foundation-outputs'
import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print("Outputs will be written to:", DRIVE_OUTPUT_DIR)

## 4. Download datasets

One-time (skips any file that already exists — cheap to re-run). ETT family comes from raw GitHub; Weather/Exchange/ECL/Traffic from HuggingFace (`thuml/Time-Series-Library`, public, no token needed).

In [ ]:
!python download_dataset.py --data_path ./data \
    --datasets ETTh1 ETTh2 ETTm1 ETTm2 Weather Exchange ECL Traffic

## 5. Configure the experiment

Presets mirror the exact HPC/Colab jobs already run, so results are directly comparable. Pick one, or edit the dict directly for something new. **`foundation_nano_sw`** (104,858 params) is the current priority — a midpoint between `medium` (254K, Weather wins but ETTh1 loses) and `tiny` (50.7K, ETTh1 improves but Weather loses -16-20%), to see where that tradeoff actually bends.

In [ ]:
PRESETS = {
    # exp_name                      : dict of train_foundation.py flags that differ from defaults
    "foundation_medium":                dict(),  # baseline: uniform sampling, everything default
    "foundation_medium_sizeweighted":   dict(size_weighted_sampling=True),
    "foundation_medium_sw_lowlr":       dict(size_weighted_sampling=True, lr=3e-4),
    "foundation_medium_sw_hyp05":       dict(size_weighted_sampling=True, hyp_hidden_scale=0.5),
    "foundation_medium_sw_proj16":      dict(size_weighted_sampling=True, proj_hidden=16),
    "foundation_medium_sw_sched20":     dict(size_weighted_sampling=True, lr=1e-4, epochs=20, warmup_epochs=2, patience=20),
    "foundation_tiny_sw":               dict(size="tiny", size_weighted_sampling=True, proj_hidden=24),  # 50.7K params: best epoch 8, val floor 0.347, but Weather zero-shot -16-20%
    "foundation_nano_sw":                dict(size="nano", size_weighted_sampling=True, proj_hidden=32),  # <-- 104,858 params, midpoint test, prioritize this
}

PRESET = "foundation_nano_sw"   # <-- change this and re-run cells 5-6 for each experiment

cfg = PRESETS[PRESET]
print(f"Running preset '{PRESET}': {cfg or '(all defaults)'}")

## 6. Run training

Builds the exact same flag set as `scripts/train_foundation.sh` from `cfg` + defaults, streams output live, and writes into the Drive-mounted output dir. If your report deadline is tight and a run isn't converging by epoch ~10-15, it's safe to interrupt (Runtime -> Interrupt execution) — the best checkpoint so far is already saved, but note in-domain/zero-shot results are only written at the very end after early stopping, so an interrupted run won't have those JSON files. To force a shorter run instead of interrupting, lower `--patience`/`--epochs` in `defaults` below.

In [ ]:
import subprocess, sys, shlex

defaults = dict(
    pretrain_datasets  = "ETTh1 ETTh2 ETTm1",
    zero_shot_datasets = "ETTm2 Weather Exchange ECL Traffic",
    seq_len       = 336,
    horizons      = "96 192 336 720",
    size          = "medium",
    patch_size    = 16,
    patch_stride  = 8,
    train_stride  = 1,
    epochs        = 50,
    warmup_epochs = 3,
    batch_size    = 32,
    lr            = 1e-3,
    weight_decay  = 1e-4,
    patience      = 15,
    num_workers   = 2,       # Colab has fewer CPU cores than the HPC node; 2 is safer than HPC's 4
    geo_dropout   = 0.2,
    hyp_hidden_scale = 1.0,
    curvature_wd  = 1e-3,
    proj_hidden   = 64,
)
merged = {**defaults, **cfg}

flag_map = {
    'pretrain_datasets': '--pretrain_datasets', 'zero_shot_datasets': '--zero_shot_datasets',
    'seq_len': '--seq_len', 'horizons': '--horizons', 'size': '--size',
    'patch_size': '--patch_size', 'patch_stride': '--patch_stride', 'train_stride': '--train_stride',
    'epochs': '--epochs', 'warmup_epochs': '--warmup_epochs', 'batch_size': '--batch_size',
    'lr': '--lr', 'weight_decay': '--weight_decay', 'patience': '--patience',
    'num_workers': '--num_workers', 'geo_dropout': '--geo_dropout',
    'hyp_hidden_scale': '--hyp_hidden_scale', 'curvature_wd': '--curvature_wd',
    'proj_hidden': '--proj_hidden',
}

cmd = [sys.executable, 'train_foundation.py']
for key, flag in flag_map.items():
    cmd += [flag] + str(merged[key]).split()
cmd += ['--data_path', './data']
cmd += ['--exp_name', PRESET]
cmd += ['--output_dir', DRIVE_OUTPUT_DIR]
if merged.get('size_weighted_sampling', False):
    cmd += ['--size_weighted_sampling']

print("Command:\n", shlex.join(cmd), "\n")

# Stream output live instead of buffering to the end
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()
print(f"\nExit code: {process.returncode}")

## 7. Inspect results

In [ ]:
import json, os

for suffix in ['indomain_results.json', 'zeroshot_results.json']:
    path = os.path.join(DRIVE_OUTPUT_DIR, f"{PRESET}_{suffix}")
    if os.path.exists(path):
        print(f"=== {suffix} ===")
        print(json.dumps(json.load(open(path)), indent=2))
        print()
    else:
        print(f"[not found yet] {path}")

## 8. (Optional) Zip everything for the report

Bundles this preset's checkpoint/log/results from Drive into a single downloadable zip.

In [ ]:
import glob, zipfile

zip_path = f"/content/{PRESET}_outputs.zip"
files = glob.glob(os.path.join(DRIVE_OUTPUT_DIR, f"{PRESET}_*"))
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in files:
        zf.write(f, arcname=os.path.basename(f))
print(f"Zipped {len(files)} files -> {zip_path}")

from google.colab import files as colab_files
colab_files.download(zip_path)

## Running more than one experiment

Change `PRESET` in section 5 and re-run sections 5-7 (skip 1-4, no need to re-clone/re-download). Since `--output_dir` points at the same Drive folder for every preset, results accumulate there across the whole session — and across sessions, since Drive persists.